# Kaggle Playground Series S6E5 — Predict F1 Pit Stops
### Author: Ruide Yin

## Stage 2: Full Pipeline — Feature Engineering & Modeling

Continues from Stage 1 (EDA). 

### 2.0 Setup & Load Data

In [1]:
import pandas as pd
import numpy as np
import os
import time
import warnings
import gc
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pickle

warnings.filterwarnings('ignore')
os.makedirs('./outputs', exist_ok=True)

SEED = 12138
N_FOLDS = 5
TARGET = 'PitNextLap'

train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Positive rate: {train[TARGET].mean():.4f}")

Train: (439140, 16), Test: (188165, 15)
Positive rate: 0.1990


### 2.0.1 Quick Check: RaceProgress Definition

In [2]:
# Verify: is RaceProgress == LapNumber / max(LapNumber) per race-driver-year group?
check = train.copy()
check['MaxLap'] = check.groupby(['Driver', 'Race', 'Year'])['LapNumber'].transform('max')
check['Computed_RP'] = check['LapNumber'] / check['MaxLap']
diff = (check['RaceProgress'] - check['Computed_RP']).abs()
print(f"Max absolute diff:  {diff.max():.6f}")
print(f"Mean absolute diff: {diff.mean():.6f}")
print(f"Exact match rate:   {(diff < 1e-6).mean():.4f}")

# Also check if it could be LapNumber / TotalRaceLaps (not per driver)
check2 = train.copy()
check2['MaxLapRace'] = check2.groupby(['Race', 'Year'])['LapNumber'].transform('max')
check2['Computed_RP2'] = check2['LapNumber'] / check2['MaxLapRace']
diff2 = (check2['RaceProgress'] - check2['Computed_RP2']).abs()
print(f"\nUsing race-level max lap:")
print(f"Max absolute diff:  {diff2.max():.6f}")
print(f"Mean absolute diff: {diff2.mean():.6f}")
print(f"Exact match rate:   {(diff2 < 1e-6).mean():.4f}")
del check, check2; gc.collect()

Max absolute diff:  0.987179
Mean absolute diff: 0.120311
Exact match rate:   0.0142

Using race-level max lap:
Max absolute diff:  0.983051
Mean absolute diff: 0.038551
Exact match rate:   0.1054


37

In [ ]:
# ── 2.0.2 Load original F1 strategy dataset (synthetic-source trick) ──
# The competition data is generated from this real-world dataset. Concatenating
# it back to train gives the model extra non-synthetic samples for free.
# Kaggle dataset: "F1 Strategy Dataset | Pit Stop Prediction"
# Add it as a Kaggle input before running this cell.

ORIGINAL_PATH = './f1_strategy_dataset_v4.csv'

original = None

original = pd.read_csv(ORIGINAL_PATH)

if original is None:
    print("WARNING: original dataset not found. Proceeding with competition data only.")
else:
    original = original.drop(columns=['Normalized_TyreLife'], errors='ignore')
    original['IsOriginalData'] = 1
    train['IsOriginalData'] = 0
    test['IsOriginalData'] = 0
    if TARGET in original.columns:
        original = original.dropna(subset=[TARGET]).reset_index(drop=True)
        original[TARGET] = original[TARGET].astype(int)
    else:
        print("WARNING: target column missing in original — discarding.")
        original = None

print(f"Train: {train.shape}, Test: {test.shape}, "
      f"Original: {None if original is None else original.shape}")

## Part 1: Feature Engineering

All features are constructed inside `build_features()` so train and test are processed identically.
Statistics that depend on the target (`PitNextLap`) are computed from train only, then merged to both.

### 2.1 Feature Engineering Functions

In [ ]:
# ── 1.1 Normalized TyreLife approximation (train-only stats) ────────────────
def compute_tyrelife_stats(train_df):
    """Compute pit-lap TyreLife quantiles from train only."""
    pit_rows = train_df[train_df[TARGET] == 1]
    stats = {}
    for group_cols, suffix in [
        (['Compound', 'Race'],   'CR'),
        (['Compound', 'Year'],   'CY'),
        (['Compound', 'Stint'],  'CS'),
        (['Compound', 'Driver'], 'CD'),
    ]:
        g = pit_rows.groupby(group_cols)['TyreLife'].quantile([0.5, 0.75]).unstack()
        g.columns = [f'TL_p50_{suffix}', f'TL_p75_{suffix}']
        g = g.reset_index()
        stats[(tuple(group_cols), suffix)] = g
    return stats

def merge_tyrelife_features(df, stats):
    for (group_cols, suffix), stat_df in stats.items():
        df = df.merge(stat_df, on=list(group_cols), how='left')
        p50_col = f'TL_p50_{suffix}'
        p75_col = f'TL_p75_{suffix}'
        df[f'TL_ratio_p50_{suffix}'] = df['TyreLife'] / df[p50_col].replace(0, np.nan)
        df[f'TL_ratio_p75_{suffix}'] = df['TyreLife'] / df[p75_col].replace(0, np.nan)
        df[f'TL_over_p75_{suffix}']  = (df['TyreLife'] > df[p75_col]).astype(np.int8)
        df.drop(columns=[p50_col, p75_col], inplace=True)
    return df

# ── 1.2 Tyre degradation features ──────────────────────────────────────────
def add_degradation_features(df, laptime_q01, laptime_q99, race_year_median):
    df['Deg_per_lap'] = np.where(df['TyreLife'] == 0, 0,
                                 df['Cumulative_Degradation'] / df['TyreLife'])
    df['LapTime_Delta_clipped'] = df['LapTime_Delta'].clip(laptime_q01, laptime_q99)
    df = df.merge(race_year_median, on=['Race', 'Year'], how='left')
    df['LapTime_dev'] = df['LapTime (s)'] - df['LapTime_median']
    df.drop(columns=['LapTime_median'], inplace=True)
    df['Abs_Cumulative_Degradation'] = df['Cumulative_Degradation'].abs()
    df['Positive_Degradation'] = (df['Cumulative_Degradation'] > 0).astype(np.int8)
    df['DeltaAbs'] = df['LapTime_Delta'].abs()
    return df

# ── 1.3 Race progress features ─────────────────────────────────────────────
def add_progress_features(df):
    df['RemainingProgress'] = 1.0 - df['RaceProgress']
    df['is_last_10pct'] = (df['RaceProgress'] >= 0.90).astype(np.int8)
    df['is_last_5pct']  = (df['RaceProgress'] >= 0.95).astype(np.int8)
    df['is_first_5pct'] = (df['RaceProgress'] <= 0.05).astype(np.int8)
    df['RacePhase'] = pd.cut(
        df['RaceProgress'],
        bins=[-np.inf, 0.20, 0.40, 0.60, 0.80, np.inf],
        labels=['P1', 'P2', 'P3', 'P4', 'P5'],
    ).astype(str)
    df['LapBin'] = pd.cut(
        df['LapNumber'],
        bins=[-np.inf, 5, 10, 20, 35, 50, np.inf],
        labels=['L_000_005', 'L_006_010', 'L_011_020', 'L_021_035', 'L_036_050', 'L_051_plus'],
    ).astype(str)
    df['TyreLifeBin'] = pd.cut(
        df['TyreLife'],
        bins=[-np.inf, 3, 7, 12, 20, 30, np.inf],
        labels=['T_000_003', 'T_004_007', 'T_008_012', 'T_013_020', 'T_021_030', 'T_031_plus'],
    ).astype(str)
    df['PositionBin'] = pd.cut(
        df['Position'],
        bins=[-np.inf, 3, 8, 14, np.inf],
        labels=['front', 'upper_mid', 'lower_mid', 'back'],
    ).astype(str)
    eps = 1e-6
    rp = df['RaceProgress'].clip(lower=eps)
    df['EstimatedTotalLaps'] = (df['LapNumber'] / rp).clip(1, 120).astype(np.float32)
    df['LapsRemaining'] = (df['EstimatedTotalLaps'] - df['LapNumber']).clip(lower=0)
    return df

# ── 1.4 Strategy features ──────────────────────────────────────────────────
COMPOUND_MAP = {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2, 'INTERMEDIATE': 3, 'WET': 4}

def add_strategy_features(df):
    df['is_last_stint'] = (df['Stint'] == df['PitStop'] + 1).astype(np.int8)
    df['Compound_ord'] = df['Compound'].map(COMPOUND_MAP).fillna(2).astype(np.int8)
    df['TyreAgeRatio']      = df['TyreLife'] / df['LapNumber'].clip(lower=1)
    df['LapPerTyreLife']    = df['LapNumber'] / (df['TyreLife'] + 1)
    df['TyreLifeMinusLap']  = df['TyreLife'] - df['LapNumber']
    df['LapMinusTyreLife']  = df['LapNumber'] - df['TyreLife']
    df['TyreAgeVsRace']     = df['TyreLife'] / df['EstimatedTotalLaps'].clip(lower=1)
    df['PitWindowPressure'] = df['TyreLife'] * df['RaceProgress']
    df['TyreLife_to_LapsRemaining'] = df['TyreLife'] / (df['LapsRemaining'] + 1)
    df['StintPressure']     = df['Stint'] * df['TyreLife']
    df['Is_First_Stint']    = (df['Stint'] == 1).astype(np.int8)
    df['Is_Late_Stint']     = (df['Stint'] >= 3).astype(np.int8)
    df['Stint_x_LapNumber'] = df['Stint'] * df['LapNumber']
    df['DegPerRaceLap']     = df['Cumulative_Degradation'] / df['LapNumber'].clip(lower=1)
    df['DegPerTyreLap']     = df['Cumulative_Degradation'] / df['TyreLife'].clip(lower=1)
    df['AbsDegPerTyreLap']  = df['Cumulative_Degradation'].abs() / df['TyreLife'].clip(lower=1)
    df['DeltaPerTyreLap']   = df['LapTime_Delta'] / df['TyreLife'].clip(lower=1)
    df['Abs_Position_Change'] = df['Position_Change'].abs()
    df['PositionPressure']  = df['Position'] * df['RaceProgress']
    return df

# ── 1.4b Race-level domain knowledge tags (multi-dimensional) ──────────────
def add_race_attributes(df):
    """Tag each race with independent dimensions: track type, temperature, speed, abrasion."""
    STREET = {
        'Monaco Grand Prix', 'Singapore Grand Prix', 'Azerbaijan Grand Prix',
        'Saudi Arabian Grand Prix', 'Las Vegas Grand Prix', 'Miami Grand Prix',
        'Australian Grand Prix',
    }
    HOT = {
        'Bahrain Grand Prix', 'Saudi Arabian Grand Prix', 'Miami Grand Prix',
        'Spanish Grand Prix', 'Singapore Grand Prix', 'Hungarian Grand Prix',
        'Qatar Grand Prix', 'Abu Dhabi Grand Prix',
    }
    COOL = {
        'British Grand Prix', 'Belgian Grand Prix', 'Las Vegas Grand Prix',
        'Dutch Grand Prix', 'Canadian Grand Prix',
    }
    HIGH_SPEED = {
        'Italian Grand Prix', 'Belgian Grand Prix', 'British Grand Prix',
        'Las Vegas Grand Prix', 'Saudi Arabian Grand Prix', 'Azerbaijan Grand Prix',
        'Mexico City Grand Prix',
    }
    LOW_SPEED = {
        'Monaco Grand Prix', 'Singapore Grand Prix', 'Hungarian Grand Prix',
        'Spanish Grand Prix',
    }
    ABRASIVE = {
        'British Grand Prix', 'Spanish Grand Prix', 'Bahrain Grand Prix',
        'Qatar Grand Prix', 'Australian Grand Prix',
    }
    SMOOTH = {
        'Monaco Grand Prix', 'Las Vegas Grand Prix', 'Russian Grand Prix',
    }
    df['is_street_race']    = df['Race'].isin(STREET).astype(np.int8)
    df['is_hot_race']       = df['Race'].isin(HOT).astype(np.int8)
    df['is_cool_race']      = df['Race'].isin(COOL).astype(np.int8)
    df['is_high_speed']     = df['Race'].isin(HIGH_SPEED).astype(np.int8)
    df['is_low_speed']      = df['Race'].isin(LOW_SPEED).astype(np.int8)
    df['is_abrasive']       = df['Race'].isin(ABRASIVE).astype(np.int8)
    df['is_smooth']         = df['Race'].isin(SMOOTH).astype(np.int8)
    df['hot_x_abrasive']    = df['is_hot_race'] * df['is_abrasive']
    df['cool_x_smooth']     = df['is_cool_race'] * df['is_smooth']
    df['street_x_low_speed']= df['is_street_race'] * df['is_low_speed']
    return df

# ── 1.5 Cross categorical features (HIGH-CARDINALITY, fed to CatBoost as cat) ──
def add_cross_features(df):
    def cross(name, cols):
        if all(c in df.columns for c in cols):
            v = df[cols[0]].astype(str)
            for c in cols[1:]:
                v = v + '_' + df[c].astype(str)
            df[name] = v
    cross('Race_Year',             ['Race', 'Year'])
    cross('Compound_Stint',        ['Compound', 'Stint'])
    cross('Driver_Race',           ['Driver', 'Race'])
    cross('Driver_Compound',       ['Driver', 'Compound'])
    cross('Race_Compound',         ['Race', 'Compound'])
    cross('Race_Compound_Stint',   ['Race', 'Compound', 'Stint'])
    cross('Compound_RacePhase',    ['Compound', 'RacePhase'])
    cross('Compound_TyreLifeBin',  ['Compound', 'TyreLifeBin'])
    cross('RacePhase_TyreLifeBin', ['RacePhase', 'TyreLifeBin'])
    return df

# ── 1.6 Target encoding (5-Fold leak-free) WITH Bayesian smoothing ─────────
def target_encode_column(train_df, test_df, col, target, n_folds=5, seed=42, alpha=10.0):
    """K-Fold leak-free TE with Bayesian smoothing.
    encoded = (count * group_mean + alpha * global_mean) / (count + alpha)
    """
    global_mean = train_df[target].mean()
    train_enc = pd.Series(np.nan, index=train_df.index, name=f'{col}_te')
    kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for trn_idx, val_idx in kf.split(train_df, train_df[target]):
        tr = train_df.iloc[trn_idx]
        agg = tr.groupby(col)[target].agg(['mean', 'count'])
        agg['smoothed'] = (agg['count'] * agg['mean'] + alpha * global_mean) / (agg['count'] + alpha)
        train_enc.iloc[val_idx] = train_df.iloc[val_idx][col].map(agg['smoothed'])
    train_enc.fillna(global_mean, inplace=True)
    full_agg = train_df.groupby(col)[target].agg(['mean', 'count'])
    full_agg['smoothed'] = (full_agg['count'] * full_agg['mean'] + alpha * global_mean) / (full_agg['count'] + alpha)
    test_enc = test_df[col].map(full_agg['smoothed']).fillna(global_mean)
    test_enc.name = f'{col}_te'
    return train_enc, test_enc

def add_encoding_features(train_df, test_df):
    for col in ['Driver', 'Race']:
        tr_te, te_te = target_encode_column(train_df, test_df, col, TARGET)
        train_df[f'{col}_te'] = tr_te.values
        test_df[f'{col}_te']  = te_te.values
    return train_df, test_df

# ── 1.7 Frequency / count encoding for categorical-ish columns ─────────────
def add_frequency_features(frames, freq_cols):
    """Pool counts across train+test+(original); add count and freq columns."""
    total = sum(len(f) for f in frames)
    for col in freq_cols:
        if not all(col in f.columns for f in frames):
            continue
        pooled = pd.concat([f[col].astype('string') for f in frames], axis=0).fillna('__MISSING__')
        counts = pooled.value_counts(dropna=False)
        for f in frames:
            keys = f[col].astype('string').fillna('__MISSING__')
            f[f'{col}_count'] = keys.map(counts).fillna(0).astype(np.int32)
            f[f'{col}_freq']  = (f[f'{col}_count'] / total).astype(np.float32)
    return frames

# ── 1.8 Group statistics (mean / std / diff_from_mean per group) ───────────
def add_group_statistics(frames):
    group_cols = ['Race_Year', 'Race_Compound_Stint', 'Driver_Race', 'Compound_Stint']
    value_cols = ['LapTime_Delta', 'Position_Change', 'RaceProgress', 'TyreLife']
    needed = list(set(group_cols + value_cols))
    pooled = pd.concat([f[[c for c in needed if c in f.columns]].copy()
                        for f in frames], axis=0, ignore_index=True)
    for gcol in group_cols:
        if gcol not in pooled.columns:
            continue
        for vcol in value_cols:
            if vcol not in pooled.columns:
                continue
            stats = pooled.groupby(gcol, dropna=False)[vcol].agg(['mean', 'std'])
            mean_col = f'{vcol}_mean_by_{gcol}'
            std_col  = f'{vcol}_std_by_{gcol}'
            diff_col = f'{vcol}_diff_mean_by_{gcol}'
            for f in frames:
                keys = f[gcol]
                f[mean_col] = keys.map(stats['mean']).astype(np.float32)
                f[std_col]  = keys.map(stats['std']).fillna(0).astype(np.float32)
                f[diff_col] = (f[vcol] - f[mean_col]).astype(np.float32)
    return frames

# ── 1.9 Interaction features ───────────────────────────────────────────────
def add_interaction_features(df):
    df['Compound_x_TyreLife']      = df['Compound_ord'] * df['TyreLife']
    df['Compound_x_CumDeg']        = df['Compound_ord'] * df['Cumulative_Degradation']
    df['RaceProgress_x_TyreLife']  = df['RaceProgress'] * df['TyreLife']
    df['Position_x_LTDelta']       = df['Position'] * df['LapTime_Delta_clipped']
    return df

# ── 1.10 Lag features (NaN preserved when laps not consecutive) ────────────
def add_lag_features(df):
    df = df.sort_values(['Driver', 'Race', 'Year', 'LapNumber']).copy()
    grp = df.groupby(['Driver', 'Race', 'Year'])
    df['LapNum_diff']      = grp['LapNumber'].diff()
    df['LapTime_lag_diff'] = grp['LapTime (s)'].diff()
    df['CumDeg_lag_diff']  = grp['Cumulative_Degradation'].diff()
    mask = df['LapNum_diff'] != 1
    df.loc[mask, 'LapTime_lag_diff'] = np.nan
    df.loc[mask, 'CumDeg_lag_diff']  = np.nan
    df.drop(columns=['LapNum_diff'], inplace=True)
    return df

### 2.2 Build Features

In [ ]:
def build_features(train_df, test_df, original_df=None):
    """Master function. Original is processed alongside train/test, then concatenated to train."""
    t0 = time.time()

    # Stats from train only (target-dependent)
    tl_stats = compute_tyrelife_stats(train_df)
    laptime_q01 = train_df['LapTime_Delta'].quantile(0.01)
    laptime_q99 = train_df['LapTime_Delta'].quantile(0.99)
    race_year_median = (train_df.groupby(['Race', 'Year'])['LapTime (s)']
                        .median().reset_index()
                        .rename(columns={'LapTime (s)': 'LapTime_median'}))

    frames_in = [train_df, test_df] + ([original_df] if original_df is not None else [])
    processed = []
    for df in frames_in:
        df = merge_tyrelife_features(df, tl_stats)
        df = add_degradation_features(df, laptime_q01, laptime_q99, race_year_median)
        df = add_progress_features(df)
        df = add_strategy_features(df)
        df = add_race_attributes(df)
        df = add_cross_features(df)
        df = add_interaction_features(df)
        df = add_lag_features(df)
        processed.append(df)

    if original_df is not None:
        train_df, test_df, original_df = processed
        train_df, test_df = add_encoding_features(train_df, test_df)
        # Map TE onto original using train's full smoothed means
        for col in ['Driver', 'Race']:
            full_agg = train_df.groupby(col)[TARGET].agg(['mean', 'count'])
            global_mean = train_df[TARGET].mean()
            full_agg['smoothed'] = (full_agg['count'] * full_agg['mean'] + 10.0 * global_mean) / (full_agg['count'] + 10.0)
            original_df[f'{col}_te'] = original_df[col].map(full_agg['smoothed']).fillna(global_mean)
        frames = add_frequency_features(
            [train_df, test_df, original_df],
            freq_cols=['Driver','Race','Compound','Race_Year','Compound_Stint',
                       'Driver_Race','Driver_Compound','Race_Compound',
                       'Race_Compound_Stint','Compound_RacePhase',
                       'Compound_TyreLifeBin','RacePhase_TyreLifeBin',
                       'LapBin','TyreLifeBin','PositionBin'],
        )
        frames = add_group_statistics(frames)
        train_df, test_df, original_df = frames
    else:
        train_df, test_df = processed
        train_df, test_df = add_encoding_features(train_df, test_df)
        frames = add_frequency_features(
            [train_df, test_df],
            freq_cols=['Driver','Race','Compound','Race_Year','Compound_Stint',
                       'Driver_Race','Driver_Compound','Race_Compound',
                       'Race_Compound_Stint','Compound_RacePhase',
                       'Compound_TyreLifeBin','RacePhase_TyreLifeBin',
                       'LapBin','TyreLifeBin','PositionBin'],
        )
        frames = add_group_statistics(frames)
        train_df, test_df = frames

    if original_df is not None:
        common = [c for c in train_df.columns if c in original_df.columns]
        missing = set(train_df.columns) - set(original_df.columns)
        for c in missing:
            original_df[c] = np.nan
        original_df = original_df[train_df.columns]
        train_df = pd.concat([train_df, original_df], axis=0, ignore_index=True)
        print(f"After concatenating original: train shape = {train_df.shape}")

    print(f"Feature engineering done in {time.time()-t0:.1f}s")
    print(f"Train: {train_df.shape}, Test: {test_df.shape}")
    return train_df, test_df

train, test = build_features(train.copy(), test.copy(),
                              original.copy() if original is not None else None)

train.to_csv('./outputs/train_fe.csv', index=False)
test.to_csv('./outputs/test_fe.csv', index=False)
print(f"Saved: train_fe.csv ({train.shape}), test_fe.csv ({test.shape})")

Feature engineering done in 3.0s
Train: (439140, 47), Test: (188165, 46)
Saved: train_fe.csv ((439140, 47)), test_fe.csv ((188165, 46))


### 2.3 Define Feature Columns & Prepare Arrays

In [ ]:
# Categorical columns to KEEP as strings for CatBoost native handling
CAT_COLS = [
    'Driver', 'Compound', 'Race',
    'Race_Year', 'Compound_Stint', 'Driver_Race', 'Driver_Compound',
    'Race_Compound', 'Race_Compound_Stint',
    'Compound_RacePhase', 'Compound_TyreLifeBin', 'RacePhase_TyreLifeBin',
    'RacePhase', 'LapBin', 'TyreLifeBin', 'PositionBin',
]
CAT_COLS = [c for c in CAT_COLS if c in train.columns and c in test.columns]

for c in CAT_COLS:
    train[c] = train[c].astype('string').fillna('__MISSING__').astype(str)
    test[c]  = test[c].astype('string').fillna('__MISSING__').astype(str)

drop_cols = ['id', TARGET]
NUM_FEATURES = [c for c in train.columns if c not in drop_cols + CAT_COLS]
ALL_FEATURES = NUM_FEATURES + CAT_COLS
print(f"Numeric: {len(NUM_FEATURES)}, Categorical: {len(CAT_COLS)}, Total: {len(ALL_FEATURES)}")

# Numeric arrays — for GBDTs, KEEP NaN (they handle natively)
X = train[NUM_FEATURES].values.astype(np.float32)
y = train[TARGET].values.astype(np.float32)
X_test = test[NUM_FEATURES].values.astype(np.float32)
test_ids = test['id'].values

# Filled versions ONLY for LR / MLP
X_filled = np.nan_to_num(X, nan=0.0)
X_test_filled = np.nan_to_num(X_test, nan=0.0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filled)
X_test_scaled = scaler.transform(X_test_filled)

# CatBoost-specific: full DataFrame with categoricals preserved
X_cat_df = train[ALL_FEATURES].copy()
X_test_cat_df = test[ALL_FEATURES].copy()
CAT_INDICES = [ALL_FEATURES.index(c) for c in CAT_COLS]

pos_count = y.sum()
neg_count = len(y) - pos_count
scale_pos = neg_count / max(pos_count, 1)

# Groups for StratifiedGroupKFold CV in stage 3
groups = (train['Race'].astype(str) + '_' + train['Year'].astype(str)).values

# Save artifacts
np.save('./outputs/X.npy', X)
np.save('./outputs/y.npy', y)
np.save('./outputs/X_test.npy', X_test)
np.save('./outputs/X_filled.npy', X_filled)
np.save('./outputs/X_test_filled.npy', X_test_filled)
np.save('./outputs/X_scaled.npy', X_scaled)
np.save('./outputs/X_test_scaled.npy', X_test_scaled)
np.save('./outputs/test_ids.npy', test_ids)
np.save('./outputs/groups.npy', groups, allow_pickle=True)
X_cat_df.to_parquet('./outputs/X_cat_df.parquet')
X_test_cat_df.to_parquet('./outputs/X_test_cat_df.parquet')
pickle.dump(NUM_FEATURES, open('./outputs/features_list.pkl', 'wb'))
pickle.dump(ALL_FEATURES, open('./outputs/all_features.pkl', 'wb'))
pickle.dump(CAT_COLS, open('./outputs/cat_cols.pkl', 'wb'))
pickle.dump(CAT_INDICES, open('./outputs/cat_indices.pkl', 'wb'))
pickle.dump(scale_pos, open('./outputs/scale_pos.pkl', 'wb'))
joblib.dump(scaler, './outputs/scaler.pkl')

# Alias for compatibility with old naming
FEATURES = NUM_FEATURES

print(f"scale_pos_weight = {scale_pos:.2f}")
print(f"Groups: {len(np.unique(groups))} unique race-year combinations")

Number of features: 42
['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'TL_ratio_p50_CR', 'TL_ratio_p75_CR', 'TL_over_p75_CR', 'TL_ratio_p50_CY', 'TL_ratio_p75_CY', 'TL_over_p75_CY', 'TL_ratio_p50_CS', 'TL_ratio_p75_CS', 'TL_over_p75_CS', 'TL_ratio_p50_CD', 'TL_ratio_p75_CD', 'TL_over_p75_CD', 'Deg_per_lap', 'LapTime_Delta_clipped', 'LapTime_dev', 'RemainingProgress', 'is_last_10pct', 'is_last_5pct', 'is_first_5pct', 'is_last_stint', 'Compound_ord', 'Driver_te', 'Race_te', 'Driver_freq', 'Race_freq', 'Compound_x_TyreLife', 'Compound_x_CumDeg', 'RaceProgress_x_TyreLife', 'Position_x_LTDelta', 'LapTime_lag_diff', 'CumDeg_lag_diff']

scale_pos_weight = 4.03


## Part 2: Baseline Modeling (StratifiedKFold 5-Fold)

All models use default or reasonable parameters. Class imbalance handled via model parameters.

### 2.4 Baseline Training Harness

In [ ]:
def train_sklearn_model(model, X_tr, y_tr, X_te, name, n_folds=N_FOLDS, use_scaled=False):
    """Generic K-Fold trainer for sklearn-API models. Returns oof_preds, test_preds, auc, elapsed."""
    t0 = time.time()
    oof = np.zeros(len(y_tr))
    test_preds = np.zeros(len(X_te))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    X_use = X_scaled if use_scaled else X_tr
    X_te_use = X_test_scaled if use_scaled else X_te

    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_use, y_tr)):
        clf = model.__class__(**model.get_params())
        clf.fit(X_use[trn_idx], y_tr[trn_idx])
        oof[val_idx] = clf.predict_proba(X_use[val_idx])[:, 1]
        test_preds += clf.predict_proba(X_te_use)[:, 1] / n_folds

    auc = roc_auc_score(y_tr, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f}  |  Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed


def train_lgb_model(params, X_tr, y_tr, X_te, name, n_folds=N_FOLDS):
    t0 = time.time()
    oof = np.zeros(len(y_tr))
    test_preds = np.zeros(len(X_te))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_tr, y_tr)):
        dtrain = lgb.Dataset(X_tr[trn_idx], y_tr[trn_idx])
        dval   = lgb.Dataset(X_tr[val_idx], y_tr[val_idx], reference=dtrain)
        model = lgb.train(params, dtrain, num_boost_round=3000,
                          valid_sets=[dval], callbacks=[
                              lgb.early_stopping(100, verbose=False),
                              lgb.log_evaluation(0)
                          ])
        oof[val_idx] = model.predict(X_tr[val_idx])
        test_preds += model.predict(X_te) / n_folds

    auc = roc_auc_score(y_tr, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f}  |  Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed


def train_xgb_model(params, X_tr, y_tr, X_te, name, n_folds=N_FOLDS):
    t0 = time.time()
    oof = np.zeros(len(y_tr))
    test_preds = np.zeros(len(X_te))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_tr, y_tr)):
        dtrain = xgb.DMatrix(X_tr[trn_idx], y_tr[trn_idx], feature_names=FEATURES)
        dval   = xgb.DMatrix(X_tr[val_idx], y_tr[val_idx], feature_names=FEATURES)
        model = xgb.train(params, dtrain, num_boost_round=3000,
                          evals=[(dval, 'val')],
                          early_stopping_rounds=100, verbose_eval=0)
        oof[val_idx] = model.predict(dval)
        dtest = xgb.DMatrix(X_te, feature_names=FEATURES)
        test_preds += model.predict(dtest) / n_folds

    auc = roc_auc_score(y_tr, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f}  |  Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed


def train_catboost_model(params, X_tr_df, y_tr, X_te_df, name, cat_indices, n_folds=N_FOLDS):
    """CatBoost with native categorical handling. X_tr_df/X_te_df are DataFrames with NaN preserved."""
    t0 = time.time()
    oof = np.zeros(len(y_tr))
    test_preds = np.zeros(len(X_te_df))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_tr_df, y_tr)):
        clf = CatBoostClassifier(**params)
        clf.fit(
            X_tr_df.iloc[trn_idx], y_tr[trn_idx],
            eval_set=(X_tr_df.iloc[val_idx], y_tr[val_idx]),
            cat_features=cat_indices,
            early_stopping_rounds=200, verbose=0,
        )
        oof[val_idx] = clf.predict_proba(X_tr_df.iloc[val_idx])[:, 1]
        test_preds += clf.predict_proba(X_te_df)[:, 1] / n_folds
    auc = roc_auc_score(y_tr, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f} | Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed

#### 2.4.1 MLP (PyTorch) Training Function

In [7]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims=(256, 128), dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_mlp_model(X_tr, y_tr, X_te, name, hidden_dims=(256, 128),
                    lr=1e-3, dropout=0.3, weight_decay=1e-4, batch_size=4096,
                    epochs=50, n_folds=N_FOLDS):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    t0 = time.time()
    oof = np.zeros(len(y_tr))
    test_preds = np.zeros(len(X_te))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    # Compute class weight
    pos_w = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
    pos_weight = torch.tensor([pos_w], dtype=torch.float32).to(device)

    X_te_t = torch.tensor(X_te, dtype=torch.float32).to(device)

    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_tr, y_tr)):
        X_t = torch.tensor(X_tr[trn_idx], dtype=torch.float32)
        y_t = torch.tensor(y_tr[trn_idx], dtype=torch.float32)
        X_v = torch.tensor(X_tr[val_idx], dtype=torch.float32).to(device)

        train_ds = TensorDataset(X_t, y_t)
        train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)

        model = MLPClassifier(X_tr.shape[1], hidden_dims, dropout).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        best_auc, patience, best_state = 0, 0, None
        for epoch in range(epochs):
            model.train()
            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)
                loss = criterion(model(xb).squeeze(), yb)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            scheduler.step()

            model.eval()
            with torch.no_grad():
                val_logits = model(X_v).squeeze().cpu().numpy()
                val_preds = 1 / (1 + np.exp(-val_logits))
                auc = roc_auc_score(y_tr[val_idx], val_preds)
            if auc > best_auc:
                best_auc = auc; patience = 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                patience += 1
                if patience >= 7:
                    break

        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            oof[val_idx] = 1 / (1 + np.exp(-model(X_v).squeeze().cpu().numpy()))
            test_preds += (1 / (1 + np.exp(-model(X_te_t).squeeze().cpu().numpy()))) / n_folds

    auc = roc_auc_score(y_tr, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f}  |  Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed, model

### 2.5 Run All Baselines

In [ ]:
results = {}  # name -> (oof, test_preds, auc, time)

# ── 2.5.1 Logistic Regression ──
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0, random_state=SEED)
oof_lr, tp_lr, auc_lr, t_lr = train_sklearn_model(lr_model, X, y, X_test, 'LR_baseline', use_scaled=True)
results['LR_baseline'] = (oof_lr, tp_lr, auc_lr, t_lr)

# ── 2.5.2 Random Forest ──
rf_model = RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_leaf=20,
                                   class_weight='balanced', random_state=SEED, n_jobs=-1)
oof_rf, tp_rf, auc_rf, t_rf = train_sklearn_model(rf_model, X_filled, y, X_test_filled, 'RF_baseline')
results['RF_baseline'] = (oof_rf, tp_rf, auc_rf, t_rf)

# ── LGB baseline: compare scale_pos_weight=1 vs scale_pos ──
lgb_params_w1 = {
    'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
    'n_jobs': -1, 'random_state': SEED,
    'learning_rate': 0.05, 'num_leaves': 63,
    'scale_pos_weight': 1.0,
}
lgb_params_wpos = {**lgb_params_w1, 'scale_pos_weight': scale_pos}

oof_lgb_w1, tp_lgb_w1, auc_lgb_w1, t_lgb_w1 = train_lgb_model(
    lgb_params_w1, X, y, X_test, 'LGB_baseline_w1'
)
oof_lgb_wpos, tp_lgb_wpos, auc_lgb_wpos, t_lgb_wpos = train_lgb_model(
    lgb_params_wpos, X, y, X_test, 'LGB_baseline_wpos'
)
LGB_BEST_SCALE_POS = scale_pos if auc_lgb_wpos > auc_lgb_w1 else 1.0
print(f"\n>>> LGB winner: scale_pos_weight={LGB_BEST_SCALE_POS:.2f}  "
      f"(w1={auc_lgb_w1:.6f}, wpos={auc_lgb_wpos:.6f})")
results['LGB_baseline'] = (
    (oof_lgb_w1, tp_lgb_w1, auc_lgb_w1, t_lgb_w1) if LGB_BEST_SCALE_POS == 1.0
    else (oof_lgb_wpos, tp_lgb_wpos, auc_lgb_wpos, t_lgb_wpos)
)

# ── XGB baseline: same comparison ──
xgb_params_w1 = {
    'objective': 'binary:logistic', 'eval_metric': 'auc',
    'tree_method': 'hist', 'device': 'cuda', 'seed': SEED,
    'learning_rate': 0.05, 'max_depth': 6,
    'scale_pos_weight': 1.0,
}
xgb_params_wpos = {**xgb_params_w1, 'scale_pos_weight': scale_pos}

oof_xgb_w1, tp_xgb_w1, auc_xgb_w1, t_xgb_w1 = train_xgb_model(
    xgb_params_w1, X, y, X_test, 'XGB_baseline_w1'
)
oof_xgb_wpos, tp_xgb_wpos, auc_xgb_wpos, t_xgb_wpos = train_xgb_model(
    xgb_params_wpos, X, y, X_test, 'XGB_baseline_wpos'
)
XGB_BEST_SCALE_POS = scale_pos if auc_xgb_wpos > auc_xgb_w1 else 1.0
print(f"\n>>> XGB winner: scale_pos_weight={XGB_BEST_SCALE_POS:.2f}  "
      f"(w1={auc_xgb_w1:.6f}, wpos={auc_xgb_wpos:.6f})")
results['XGB_baseline'] = (
    (oof_xgb_w1, tp_xgb_w1, auc_xgb_w1, t_xgb_w1) if XGB_BEST_SCALE_POS == 1.0
    else (oof_xgb_wpos, tp_xgb_wpos, auc_xgb_wpos, t_xgb_wpos)
)

pickle.dump({
    'LGB_BEST_SCALE_POS': LGB_BEST_SCALE_POS,
    'XGB_BEST_SCALE_POS': XGB_BEST_SCALE_POS,
}, open('./outputs/scale_pos_winners.pkl', 'wb'))

# CatBoost baseline — proper categorical handling, balanced via auto_class_weights
cat_params_baseline = {
    'iterations': 5000,
    'learning_rate': 0.018,
    'depth': 8,
    'l2_leaf_reg': 8.5,
    'random_strength': 0.65,
    'bootstrap_type': 'Bayesian',
    'bagging_temperature': 0.45,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'auto_class_weights': 'Balanced',
    'task_type': 'GPU',     # set 'CPU' if no GPU
    'random_seed': SEED,
    'allow_writing_files': False,
    'verbose': 0,
}
oof_cat, tp_cat, auc_cat, t_cat = train_catboost_model(
    cat_params_baseline, X_cat_df, y, X_test_cat_df,
    'CAT_baseline', CAT_INDICES, n_folds=N_FOLDS
)
results['CAT_baseline'] = (oof_cat, tp_cat, auc_cat, t_cat)

# ── 2.5.6 MLP ──
oof_mlp, tp_mlp, auc_mlp, t_mlp, mlp_model = train_mlp_model(
    X_scaled.astype(np.float32), y, X_test_scaled.astype(np.float32), 'MLP_baseline')
results['MLP_baseline'] = (oof_mlp, tp_mlp, auc_mlp, t_mlp)

[LR_baseline] OOF AUC = 0.881202  |  Time = 49.9s
[RF_baseline] OOF AUC = 0.938378  |  Time = 377.9s
[LGB_baseline] OOF AUC = 0.948450  |  Time = 129.7s
[XGB_baseline] OOF AUC = 0.948580  |  Time = 30.8s
[CAT_baseline] OOF AUC = 0.947685  |  Time = 92.9s
[MLP_baseline] OOF AUC = 0.941555  |  Time = 777.4s


In [9]:
# Print baseline summary
print("\n" + "="*60)
print("BASELINE RESULTS")
print("="*60)
baseline_df = pd.DataFrame({
    'Model': list(results.keys()),
    'OOF_AUC': [v[2] for v in results.values()],
    'Time_s': [v[3] for v in results.values()],
}).sort_values('OOF_AUC', ascending=False).reset_index(drop=True)
print(baseline_df.to_string(index=False))


BASELINE RESULTS
       Model  OOF_AUC     Time_s
XGB_baseline 0.948580  30.805804
LGB_baseline 0.948450 129.654911
CAT_baseline 0.947685  92.901379
MLP_baseline 0.941555 777.380629
 RF_baseline 0.938378 377.944832
 LR_baseline 0.881202  49.939114


In [10]:
# ── Save everything for Notebook 3 ──
np.save('./outputs/X.npy', X)
np.save('./outputs/y.npy', y)
np.save('./outputs/X_test.npy', X_test)
np.save('./outputs/X_scaled.npy', X_scaled)
np.save('./outputs/X_test_scaled.npy', X_test_scaled)
np.save('./outputs/X_filled.npy', X_filled)
np.save('./outputs/X_test_filled.npy', X_test_filled)
np.save('./outputs/test_ids.npy', test_ids)
pickle.dump(results, open('./outputs/baseline_results.pkl', 'wb'))
pickle.dump(FEATURES, open('./outputs/features_list.pkl', 'wb'))
pickle.dump(float(scale_pos), open('./outputs/scale_pos.pkl', 'wb'))
joblib.dump(scaler, './outputs/scaler.pkl')
print("Notebook 2 done. All saved to ./outputs/")

Notebook 2 done. All saved to ./outputs/
